# 06. Animation: chính sách khuyến nghị so với `all_on` có ngân sách (T11.1)

Theo `docs/huong-dan-animation-chinh-sach.md` (mục 4–6, 8): hai lượt chạy **cùng seed, cùng B**, log đầy đủ, 10 seed; ba bản đồ lục giác (Δ request, Δ chuyến hoàn thành theo **slot đặt xe**, slack dưới chính sách mới với ô tắt gạch chéo), hàng dưới là ΔN cộng dồn (trung bình và dải 95% qua seed); hai bản đồ Δ chung một thang màu đối xứng cố định (PuOr, an toàn cho người mù màu); mỗi khung một câu chú thích; khung cuối dừng 2 giây.

Cặp chính sách: **W0** `all_on_B` → tầng vùng `city` θ̂_B = 1,6 + τ̂(x)/1.000đ DR (khuyến nghị T-47 h); **W1** `all_on_B` → tầng vùng `ring1` θ̂_B = 2,15 + τ̂(x)/1.000đ DR. Dữ liệu `runs/s10/w0/anim_pair_full` (238 s) và `runs/s10/w1/h2_slots_full` (428 s; cùng bảng dùng cho kiểm H2). Lệnh ở `docs/datasets.md`; `check_dataset` OK trên 20 + 30 thư mục seed. Code: `analysis/animation.py` (T10.10, test `tests/test_animation.py`).


In [ ]:
from analysis.notebook import require_runs, setup
from analysis.plots import save_figure

plt = setup()
pair_w0, pair_w1 = require_runs("runs/s10/w0/anim_pair_full", "runs/s10/w1/h2_slots_full")


In [ ]:
from IPython.display import Image, display
import pandas as pd

from analysis import animation

def run_dirs(table, prefix):
    return sorted(d for d in (table / "runs").iterdir() if d.name.startswith(prefix))

pairs = {
    "W0": animation.load_pairs(run_dirs(pair_w0, "evaluate-efaafc260352-all_on-nan"), run_dirs(pair_w0, "evaluate-9005122b8cea-threshold-1.6")),
    "W1": animation.load_pairs(run_dirs(pair_w1, "evaluate-7c5c6b7eef3a-all_on-nan"), run_dirs(pair_w1, "evaluate-19b9276d09a6-threshold-2.15")),
}
for w, pair in pairs.items():
    totals = animation.check_totals(pair)
    assert (totals["delta_N"] == totals["delta_from_orders"]).all()     # guide §8: Σ Δ over the window = N(new) − N(old)
    wd = animation.window_delta(pair)
    slots = animation.frame_slots(pair, 5, 9)
    print(f"{w}: {wd['n_seeds']} seed, B = {pair.budget:,.0f} đ/kỳ; ΔN cả cửa sổ = {wd['mean']:.1f} ± {wd['half95']:.1f} (95%); "
          f"ΔN trong khung 5h–9h = {pair.d_done[slots].sum():.1f}; tổng Δ theo slot khớp policy_results: OK")


## 1. W0, khung 5h–9h

`docs/figures/06_anim_w0_5_9h.gif`. Câu kiểm tra của mentor ("voucher lúc 7h sáng có giúp tăng chuyến không?"): trong khung 5h–9h chính sách mới **tắt voucher ở phần lớn ô** (slack thấp) nên request giảm, số chuyến gần như không đổi (ΔN trong khung ≈ −34); phần lợi +1.479 ± 49 chuyến/ngày của chính sách mới đến **sau 9h**, khi `all_on_B` đã tiêu hết B của kỳ còn chính sách mới vẫn phát cho rider có τ̂/1.000đ cao ở ô còn xe.


In [ ]:
display(Image(filename="docs/figures/06_anim_w0_5_9h.gif"))

## 2. W1, khung 5h–9h

`docs/figures/06_anim_w1_5_9h.gif`. Cùng câu chuyện: ΔN trong khung −62, cả ngày +1.208 ± 59.


In [ ]:
display(Image(filename="docs/figures/06_anim_w1_5_9h.gif"))

## 3. W1, khung có sự kiện (21h–24h, 3 ngày có sự kiện theo `meta/shocks`)

`docs/figures/06_anim_w1_su_kien_21_24h.gif`: chỉ 3 trong 10 seed có sự kiện trong cửa sổ (seed 1: ô 0, 22:17–23:17, cầu ×20; seed 2 và 5: ô 39, 21:30–22:30 và 22:23–23:23, cầu ×2), nên bản đồ Δ là trung bình **3 seed** (bản đồ slack vẽ seed 2) và dải 95% rộng hơn; không ép sự kiện bằng `--set`. Trong khung này chính sách mới hơn `all_on_B` +180 chuyến (3 seed) — ô sự kiện và vành 1 bị tắt voucher (gạch chéo) khi slack tụt, request bớt kéo vào nơi không còn xe.


In [ ]:
event_seeds = [1, 2, 5]
pair_ev = animation.load_pairs([d for d in run_dirs(pair_w1, "evaluate-7c5c6b7eef3a-all_on-nan") if int(d.name.rsplit("-", 1)[1]) in event_seeds],
                               [d for d in run_dirs(pair_w1, "evaluate-19b9276d09a6-threshold-2.15") if int(d.name.rsplit("-", 1)[1]) in event_seeds],
                               display_seed_index=1)
slots_ev = animation.frame_slots(pair_ev, 21, 24)
print(f"3 seed có sự kiện: ΔN trong khung 21h–24h = {pair_ev.d_done[slots_ev].sum():.1f}; cả cửa sổ = {animation.window_delta(pair_ev)['mean']:.1f}")
display(Image(filename="docs/figures/06_anim_w1_su_kien_21_24h.gif"))


## Chú thích cho slide

- *Trong mô phỏng này* (quy mô Hà Nội, 61 ô H3, B = 0,3 × chi của `all_on`): chính sách khuyến nghị hơn `all_on` có cùng ngân sách **+1.479 ± 49 chuyến/ngày (W0, 10 seed)** và **+1.208 ± 59 (W1, 10 seed)**.
- Buổi sáng 5h–9h chính sách mới **không** thêm chuyến (thậm chí bớt 30–60): nó cắt voucher ở ô căng và để dành ngân sách; phần lợi đến từ việc rải chi tiêu cả ngày và chọn rider theo hiệu ứng mỗi 1.000đ.
- Mọi hiệu số ghép cặp theo seed (CRN); Δ chuyến tính theo slot đặt xe; tổng Δ trên cả cửa sổ bằng đúng N(π mới) − N(π cũ) của `results/policy_results` (kiểm ở ô code đầu).
